# Lab A &mdash; Work

In this computer exercise we will look at
<ul>
    <li>Matrix computations in NumPy and Python</li>
    <li>Homography estimation using DLT</li>
    <li>The SVD profile and the number of solutions</li>
    <li>Geometric vs algebraic errors</li>
</ul>

<!---
### Handing in

As with the preparatory questions, each student should hand in his/her notebook in Lisam.
Before uploading your `.ipynb` file, make sure that all cells can be executed in sequence.
This can be checked by selecting "Kernel" -> "Restart & Run All" from the menu.
--->

<!---
### A note on the TeacherTokens (4 in total in this computer exercise)

At some places (marked clearly in <span style="color:red;font-weight:bold;">red</span>) in the computer exercise you will be informed that you need a TeacherToken.
A TeacherToken is a short unique key on the form LAB1-XXXX-XXXX, where each X is either a digit or an uppercase letter.
You obtain TeacherTokens from one of the teachers by discussing your work on the relevant section with the teacher.
TeacherTokens must be added to the `teacher_tokens` set, and are automatically validated by the autograder.
If, for example, you have received the TeacherToken LAB1-7A3J-1PHS, you must write `teacher_tokens.add('LAB1-7A3J-1PHS')` in a code cell.
**If you work in pairs, each of you you needs a set of unique TeacherTokens.**
--->

### A note on mandatory teacher interaction
At some places (marked clearly in <span style="color:red;font-weight:bold;">red</span>) in the computer exercise you will be informed that teacher interaction is required.
Typically, you are expected to briefly explain your observations and reasoning for that section to the teacher.
Once you reach a point where teacher interaction is required, reflect a little on what occurred in that section, and then notify a teacher.
You do not have to just sit around and wait for the teacher interaction before you continue, e.g. if all teachers are busy, but do not postpone all interaction to the end of the lab.

In [ ]:
### Import python modules for linear algebra (numpy) and plotting (matplotlib).

import numpy as np
from matplotlib import pyplot as plt

%matplotlib inline
np.set_printoptions(precision = 2, suppress = True)

## 1. Lightning intro to Python and NumPy arrays

In this computer exercise, we will use [Python](https://docs.python.org/3/tutorial/), [NumPy](https://numpy.org/doc/stable/user/quickstart.html), and [Matplotlib](https://matplotlib.org/stable/tutorials/introductory/pyplot.html).
It is recommended that you scroll through the links for NumPy and Matplotlib for a while before starting the computer exercise, and then keep the tabs open for reference and inspiration.
In the following few cells are some examples of basic operations on NumPy arrays, that you are likely to need in this exercise.

In [ ]:
# Matrices are represented in NumPy using np.array, and can be entered elementwise as:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f'A == \n{A}\n')

# Addition of matrices, and multiplication by a scalar work as expected,
# so we do not provide an example. You can make one up and try.

# We can access the dimensions of a matrix A through A.shape, which is a tuple:
print(f'A.shape is {A.shape}, meaning that A is a {A.shape[0]}-by-{A.shape[1]} matrix\n')

# The transpose of A is written A.T:
print(f'A.T == \n{A.T}\n')

# We can work with blocks (slices) or single entries of matrices (note that Python is zero-indexed!):
A[:,1:] = np.array([[-1,2],[-3,4]])
# Change one entry:
A[1, 1] = -5
print(f'A == \n{A}\n')

In [ ]:
# NumPy has an annoying tendency to forget dimensions, especially
# of things we expect to be n-by-1, such as columns (also rows, which should be 1-by-n):
print(f'A[:, 0] == {A[:, 0]} has dimensions {A[:, 0].shape}, not (2, 1) as one would expect.',
      'Avoid indexing with single columns!\n')

# There are various ugly workarounds if you want column k (k=0 in the examples).
#   (i) Select the range k:k+1 instead, as in:
print(f'A[:, 0:1] == \n{A[:, 0:1]}\nhas dimensions {A[:, 0:1].shape}.\n')
#  (ii) Select a list [k] instead, as in:
print(f'A[:, [0]] == \n{A[:, [0]]}\nhas dimensions {A[:, [0]].shape}.\n')
# (iii) Add a new dimension with None:
print(f'A[:, 0, None] == \n{A[:, 0, None]}\nhas dimensions {A[:, 0, None].shape}.\n')

# Matrices can sometimes be appended to form a larger matrix using np.append, but NumPy tends
# to forget the dimensions unless you specify the direction using axis = 0 or axis = 1:
print(f'np.append(A, 2 * A, axis = 0) ==\n{np.append(A, 2 * A, axis = 0)}.\n')

In [ ]:
# You can create identity matrices using np.eye,
# matrices filled with ones with np.ones,
# zero matrices using np.zeros:
I3 = np.eye(3)
B = np.ones((2, 2))
print(f'I3 == \n{I3}\n')
print(f'B == \n{B}\n')

# Matrices with compatible dimensions can be multiplied using the operator @:
print(f'B @ A == \n{B @ A}\n')
print(f'A @ A.T == \n{A @ A.T}\n')

# Systems of equations Mx = b can be solved using the pseudo-inverse:
M = np.array([[1, 2],
              [3, 4]])
b = np.array([[0], [2]])
x = np.linalg.pinv(M) @ b
print(f'x == \n{x}\n')

In [ ]:
# The Kronecker product can be computed using np.kron:
print(f'np.kron(B, A) == \n{np.kron(B, A)}\n')

# Vectorisation of a matrix A can be done through A.reshape((-1, 1), order = 'F').
# To simplify things, we define a function vec(X) for this:
def vec(X):
    """takes a matrix X and returns the vectorisation (column stacking) of X."""
    return X.reshape((-1, 1), order = 'F')

print(f'vec(A) == \n{vec(A)}')

In [ ]:
### Define some utility functions.

def cart_to_hom(y):
    """takes an array with Cartesian coordinates as its columns,
        and returns an array with a homogeneous representation."""
    return np.append(y, np.ones((1, y.shape[1])), axis = 0)

def norm_P(y):
    """takes an array with homogeneous coordinates as its columns,
        and returns an array with P-normalised homogeneous coordinates
        as its columns."""
    return y / np.tile(y[-1, :], (y.shape[0], 1))

def norm_D(l):
    """takes an array with dual homogeneous coordinates as its columns,
        and returns an array with D-normalised dual homogeneous cordinates
        as its columns."""
    normal_norms = np.tile(np.sqrt(np.power(l[:-1, :], 2).sum(axis = 0)), (l.shape[0], 1))
    last_signs = np.tile(np.sign(l[-1, :]), (l.shape[0], 1))
    return -last_signs * l / normal_norms

def crossm(v):
    """takes a vector v of lenght 3, and returns the cross product matrix of v."""
    assert np.prod(v.shape) == 3
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]],[-v[1], v[0], 0]])

def vec(X):
    """takes a matrix X and returns the vectorisation (column stacking) of X."""
    return X.reshape((-1, 1), order = 'F')

## 2. Extracting a homography from a DLT system

Consider six point correspondences $\mathbf{y}_k \leftrightarrow \mathbf{y}_k'$, defined by
$$
    \mathbf{Y} =
    \begin{pmatrix}
        \color{blue}{\mathbf{y}_1} & \ldots & \mathbf{y}_6
    \end{pmatrix}
    =
    \begin{pmatrix}
        \color{blue}{0} & 1 & 1 & 2 & 1 & 0 \\
        \color{blue}{0} & 2 & 4 & 1 & 0 & 1 \\
        \color{blue}{1} & 1 & 1 & 1 & 1 & 1 \\
    \end{pmatrix}
$$
and
$$
    \mathbf{Y}' =
    \begin{pmatrix}
        \color{blue}{\mathbf{y}_1'} & \ldots & \mathbf{y}_6'
    \end{pmatrix}
    =
    \begin{pmatrix}
        \color{blue}{1} & -3 & -2 & 2 & 0 & -4 \\
        \color{blue}{-2} & -1 & -1 & 0 & -1 & 0 \\
        \color{blue}{1} & 1 & 1 & 1 & 1 & 1 \\
    \end{pmatrix}
    ,
$$

where the points involved in the correspondence $\mathbf{y}_1 \leftrightarrow \mathbf{y}_1'$ have been coloured in blue only to help visualisation.
The matrices $\mathbf{Y}$ and $\mathbf{Y}'$ are entered in the code cell below as `Y` and `Yprime`, respectively.

In fact, these six point correspondences are related through a homography $\mathbf{H}$, such that $\mathbf{y}_k' \sim \mathbf{H}\mathbf{y}_k$ for $k = 1,\ldots,6$.
As we have seen in the lectures, $\mathbf{H}$ can be found by computing the null space of the data matrix $\mathbf{A}$ formed from the *Direct Linear Transformation* (DLT) constraints, since $\mathbf{A}\operatorname{vec}{\mathbf{H}} = \mathbf{0}$.
The data matrix in this particular example is entered as `A` in the code cell below.


### Task: use the inhomogeneous method to find $\mathbf{H}_\text{inh}$ from the DLT system.

<ul>
    <li>Recall that the inhomogeneous method is based on rewriting $\mathbf{A}\mathbf{z} = \mathbf{0}$ as $\begin{pmatrix} \mathbf{A}_0 & \mathbf{b} \end{pmatrix}\begin{pmatrix} \mathbf{z}_0 \\ 1 \end{pmatrix} = \mathbf{0}$ and solving for $\mathbf{z}_0$.</li>
    <li>You will likely find <code>np.linalg.pinv</code> useful here.</li>
    <li>You will likely find <code>np.append</code> and <code>np.reshape</code> useful here to form $\mathbf{H}_\text{inh}$ from $\mathbf{z}_0$.</li>
    <li>By default, <code>np.reshape</code> puts the values along <i>rows</i>, but you can tell it to put them along columns instead by supplying <code>order = 'F'</code>.</li>
</ul>

In [ ]:
### Define point correspondences and the data matrix.

Y = cart_to_hom(np.array([[0, 1, 1, 2, 1, 0],
                          [0, 2, 4, 1, 0, 1]]))

Yprime = cart_to_hom(np.array([[1, -3, -2, 2, 0, -4],
                               [-2, -1, -1, 0, -1, 0]]))

A = np.array([[0, 0, 0, 0, 0, 0, 0, -1, -2],
              [0, 0, 0, 0, 0, 0, 1, 0, -1],
              [0, -1, -1, 0, -2, -2, 0, -1, -1],
              [1, 0, 3, 2, 0, 6, 1, 0, 3],
              [0, -1, -1, 0, -4, -4, 0, -1, -1],
              [1, 0, 2, 4, 0, 8, 1, 0, 2],
              [0, -2, 0, 0, -1, 0, 0, -1, 0],
              [2, 0, -4, 1, 0, -2, 1, 0, -2],
              [0, -1, -1, 0, 0, 0, 0, -1, -1],
              [1, 0, 0, 0, 0, 0, 1, 0, 0],
              [0, 0, 0, 0, -1, 0, 0, -1, 0],
              [0, 0, 0, 1, 0, 4, 1, 0, 4]])


# Hinhom = ...?
# YOUR CODE HERE
raise NotImplementedError()

print('Hinhom @ Y:')
print(norm_P(Hinhom @ Y))
print('\nYprime:')
print(Yprime)

In [ ]:
# We can also plot the points and see that they overlap:
plt.figure()
plt.plot(norm_P(Hinhom @ Y)[0, :], norm_P(Hinhom @ Y)[1, :], 'ro', mfc = 'None', mec = 'r')
plt.plot(Yprime[0, :], Yprime[1, :], 'b.')
plt.legend(['mapped points$', 'actual points'])
plt.show()

In [ ]:
# Verify that the points are mapped correctly (to high precision).
assert np.linalg.norm(Yprime - norm_P(Hinhom @ Y)) < 1e-10

### Task: use the singular value decomposition to find $\mathbf{H}_\text{svd}$ from the DLT system.

<ul>
    <li>As this is something that will be done repeatedly, you may want to create a function for this (optional).</li>
    <li>Be careful with the transposes! Note that <code>np.linalg.svd</code> returns $\mathbf{V}^\top$, not $\mathbf{V}$.</li>
    <li>You will likely find <code>np.reshape</code> useful here, to reshape $\operatorname{vec}{\mathbf{H}_\text{svd}}$ back to $\mathbf{H}_\text{svd}$.</li>
    <li>Do you obtain the same homography using the two methods, i.e., is $\mathbf{H}_\text{inh} \sim \mathbf{H}_\text{svd}$?</li>
    <li><b>Note:</b> <span style="color:red;font-weight:bold;">Explain your observations to a teacher!</span></li>
</ul>

In [ ]:
# Hsvd = ...?
# YOUR CODE HERE
raise NotImplementedError()

print('Hsvd @ Y:')
print(norm_P(Hsvd @ Y))
print('\nYprime:')
print(Yprime)

In [ ]:
# Verify that the points are mapped correctly (to high precision).
assert np.linalg.norm(Yprime - norm_P(Hsvd @ Y)) < 1e-10

## 3. Number of solutions, a taste of SVD profile

The points in Section 2 were mapped to each other *exactly* by the homography.
There could potentially exist more than one homography which accomplishes this, even when taking the homogeneity of $\mathbf{H}$ into account.

In general, to be able to say something intelligent about the solution space, we can look at the singular values of the data matrix.
The configuration of their relative sizes is referred to as the *SVD profile* of the data matrix.
The SVD profile of the data matrix in Section 2 is plotted in the following cell (we plot the $\log_{10}$ of the singular values to make the plot more readable).

### Task: explain to a lab assistant how the SVD profile relates to the nullspace (and hence to the solutions)

<ul>
    <li>With six <i>exact</i> point correspondences, what do you expect $\operatorname{rank}{\mathbf{A}}$ to be?</li>
    <li>How many solutions do we have in this particular case?</li>
    <li>Considering how the rank relates to the singular values, are you surprised by the SVD profile in the plot below?</li>
    <li>Could the rank be greater? In what situations?</li>
    <li>Could the rank be smaller? In what situations?</li>
    <li><b>Note:</b> <span style="color:red;font-weight:bold;">Explain your reasoning to a teacher!</span></li>
</ul>

In [ ]:
plt.figure()
plt.plot(np.arange(1, 10), np.log10(S), 'bo')
plt.legend(['log10(s_j)'])
plt.show()

## 4. Autogenerating the DLT system from point correspondences

Forming the DLT system by hand is error-prone drudgery, so we want a function that autogenerates it for us from point correspondences we provide.
As we have mentioned before in the course, since $\operatorname{rank}\big(\mathbf{y}_k^\top \otimes [\mathbf{y}_k']_\times\big) = 2$, we only have to use two rows to form $\mathbf{A}_k$.
For *proper points*, it is safe to use the first two rows of $\mathbf{y}_k^\top \otimes [\mathbf{y}_k']_\times$.
For *ideal points* the first two rows may be linearly dependent, and to be safe in this case, we use all three rows (we do not know in advance which row to remove).


### Task: implement the `dlt_system` function.

In [ ]:
def dlt_system(Y, Yprime, full_system = False):
    """Input:
        | Y - a 3-by-n array with homogeneous coordinates as its columns
        | Yprime - a 3-by-n array with homogeneous coordinates as its columns
        | full_system - boolean, False by default

        Output:
        | if full_system is False - a 2n-by-9 data matrix for DLT
        | if full_system is True - a 3n-by-9 data matrix for DLT"""

    assert Y.shape[0] == 3
    assert np.all(Y.shape == Yprime.shape)

    nbr_points = Y.shape[1]
    nbr_rows_per_point = (2 + full_system)
    nbr_rows = nbr_rows_per_point * nbr_points
    A = np.zeros((nbr_rows, 9))

    # YOUR CODE HERE
    raise NotImplementedError()

    return A


# If your implementation is correct, this should print the data matrix A from Section 2.
print(dlt_system(Y, Yprime))

In [ ]:
# Verify that dlt_system gives the desired result.
assert np.linalg.norm(A - dlt_system(Y, Yprime)) < 1e-10

## 5. Transforming random points, SVD profile

We will now consider homography estimation using noisy point correspondences.
This will allow us to see how the SVD profile behaves when we do not have exact correspondences.

### Task: vary the number of points and investigate what happens to the point mapping and the SVD profile

<ul>
    <li>Let $n$ be the number of point correspondences we use. For which values of $n$ is the mapping good?</li>
    <li>Can we tell from the SVD profile whether the homography mapping will be good or bad?</li>
    <li><b>Note:</b> <span style="color:red;font-weight:bold;">Explain your reasoning to a teacher!</span></li>
</ul>

In [ ]:
# Choose the number of correspondences to use (n between 1 and 9, inclusive).
n = 6


# Create some noisy point correspondences.
Y = cart_to_hom(np.array([[0, 1, 1, 2, 1, 0, 0.5, 1.5, 0.5],
                          [0, 2, 4, 1, 0, 1, 0.5, 0.5, 1.5]]))[:,:n]

Yprime = cart_to_hom(np.array([[1, -3, -2, 2, 0, -4, 4, 2/3, -10/3],
                               [-2, -1, -1, 0, -1, 0, -2, -2/3, -2/3]]))[:,:n]


# Set noise level, and add random noise to the first two coordinates
sigma = 0.1

Z = Y + sigma * np.random.randn(*Y.shape)
Z[2, :] = 1

Zprime = Yprime + sigma * np.random.randn(*Y.shape)
Zprime[2, :] = 1


# Create the DLT system and form H using the singular value decomposition.
# H = ...?
# YOUR CODE HERE
raise NotImplementedError()

# Plot the actual points and the mapped points,
# as well as the SVD profile of the data matrix.
Zmapped = norm_P(H @ Z)
plt.figure()
plt.subplot(1, 2, 1)
plt.plot(Zmapped[0, :], Zmapped[1, :], 'ro', mfc = 'None', mec = 'r')
plt.plot(Zprime[0, :], Zprime[1, :], 'b.')
plt.legend(['mapped points', 'actual points'])
plt.subplot(1, 2, 2)
plt.plot(np.log10(S), 'bo')
plt.legend(['log10(s_j)'])
plt.show()

### 5.1. Hartley normalisation

When we use the SVD to solve a least-squares problem, it will treat errors equally in all coordinates (since it works with the Euclidean norm).
This is problematic for homogeneous coordinates.
For instance, if we have two estimates,
$$
\mathbf{y}_1 = (1001,1000,1), \qquad
\mathbf{y}_2=(1000,1000,2),
$$
of a point $\mathbf{y} = (1000,1000,1)$, then $\lVert\mathbf{y}_1-\mathbf{y}\rVert = \lVert\mathbf{y}_2-\mathbf{y}\rVert = 1$, so they are equally good as measured in the Euclidean norm.
However, it is clear that $\mathbf{y}_2$ is much worse if considered as homogeneous coordinates, since $\mathbf{y}_2 \sim (500,500,1)$ has a *huge* error.

For this reason, Hartley suggested a normalisation scheme for point sets before they are used in estimation that uses homogeneous coordinates.
What *Hartley normalisation* does is to centre the points by subtracting their average, and then applying a uniform scaling of space so that the points have $\sqrt{2}$ (in 2D) as their their average distance to the origin.
This is implemented in the next cell (cf. Algorithm 13.1 on page 235 in <i><a href="https://www.cvl.isy.liu.se/research/publications/IREG/0.40/">IREG (version 0.40)</a></i>, but note a **typo**: in step 7, the translation part should also be scaled with $\sqrt{2}\big/d$).

In [ ]:
def hartley_normalise(y):
    """takes an array with homogeneous coordinates as its columns,
        and returns an array with Hartley-normalised points and
        the 3x3 transformation matrix that brings the point in y
        into the normalised points."""

    # Ensure the points are P-normalised
    ynormalised = norm_P(y)

    # Compute the average of the points
    yaverage = ynormalised.mean(axis = 1)[:, None]

    # Subtract the average from all points
    ycentered = ynormalised - np.kron(np.ones((1, ynormalised.shape[1])), yaverage)

    # Compute the scaling and rescale all points
    d = np.sqrt((ycentered ** 2).sum(axis = 0)).mean()
    yhartley = (np.sqrt(2) / d) * ycentered
    yhartley[-1, :] = 1

    # Create the transformation matrix T
    T = np.diag([np.sqrt(2) / d, np.sqrt(2) / d, 1])
    T[:2, -1:] = -(np.sqrt(2) / d) * yaverage[:2, :]
    
    return (yhartley, T)


# Compute Hartley normalised points and normalising transformations
(Zhartley, T) = hartley_normalise(Z)
(Zprimehartley, Tprime) = hartley_normalise(Zprime)

# Create the DLT system and compute Hhartley that works on the Hartley-normalised coordinates.
# Hhartley = ...?
# YOUR CODE HERE
raise NotImplementedError()

# Plot the actual points and the mapped points,
# as well as the SVD profile of the data matrix.
Zmapped = norm_P(np.linalg.inv(Tprime) @ Hhartley @ T @ Z)
plt.figure()
plt.subplot(1, 2, 1)
plt.plot(Zmapped[0, :], Zmapped[1, :], 'ro', mfc = 'None', mec = 'r')
plt.plot(Zprime[0, :], Zprime[1, :], 'b.')
plt.legend(['mapped points', 'actual points'])
plt.subplot(1, 2, 2)
plt.plot(np.log10(S), 'bo')
plt.legend(['log10(s_j)'])
plt.show()

## 6. Algebraic error vs geometric error

The homography estimation methods that we consider in this lab estimate the homography by solving a linear least-squares problem.
This is corresponds to minimising an *algebraic cost function*, and in general, this algebraic error does not tell us how well the transformation fits *geometrically*.
(When we have *exact* point correspondences, this does not matter since we will find the exact solution when we minimise the cost function, and both the algebraic and geometric errors are zero.)

If we have two candidate homography estimates, and want to decide which is the better of the two, we almost always want to compare their geometric error given by

$$
\varepsilon_\text{G}(\mathbf{H};\mathbf{y}_1,\ldots,\mathbf{y}_n,\mathbf{y}_1',\ldots,\mathbf{y}_n')
= \sum_{k=1}^n d_\text{PP}(\mathbf{y}_k',\mathbf{H}\mathbf{y}_k)^2 + d_\text{PP}(\mathbf{y}_k,\mathbf{H}^{-1}\mathbf{y}_k')^2.
$$

### Task: implement the geometric error, and compare it to the algebraic error

<ul>
    <li>Why is it meaningful to include both $d_\text{PP}(\mathbf{y}_k',\mathbf{H}\mathbf{y}_k)^2$ and $d_\text{PP}(\mathbf{y}_k,\mathbf{H}^{-1}\mathbf{y}_k')^2$ in the sum?</li>
    <li><b>Note:</b> <span style="color:red;font-weight:bold;">Explain your reasoning to a teacher!</span></li>
</ul>

In [ ]:
### Define some noisy point correspondences.
Y = cart_to_hom(np.array([[0, 1, 1, 2, 1, 0, 0.5, 1.5, 0.5],
                          [0, 2, 4, 1, 0, 1, 0.5, 0.5, 1.5]]))[:,:n]

Yprime = cart_to_hom(np.array([[1, -3, -2, 2, 0, -4, 4, 2/3, -10/3],
                               [-2, -1, -1, 0, -1, 0, -2, -2/3, -2/3]]))[:,:n]


# Set noise level, and add random noise to the first two coordinates
sigma = 0.1

Z = Y + sigma * np.random.randn(*Y.shape)
Z[2, :] = 1

Zprime = Yprime + sigma * np.random.randn(*Y.shape)
Zprime[2, :] = 1


# Define the geometric error function
def geometric_error(H, Y, Yprime):
    """returns the geometric error for the homography H with the point correspondences in Y and Yprime"""
    # YOUR CODE HERE
    raise NotImplementedError()


# Estimate a homography H from the noisy points
# H = ...?
# YOUR CODE HERE
raise NotImplementedError()

Zmapped = norm_P(np.linalg.inv(H) @ Zprime)
Zprimemapped = norm_P(H @ Z)

# Plot the points and the transformed points
plt.figure()
plt.subplot(1, 2, 1)
plt.plot(Z[0, :], Z[1, :], 'ro', mfc = 'None', mec = 'r')
plt.plot(Zmapped[0, :], Zmapped[1, :], 'b.')
plt.legend(['actual points', 'reprojected points'])
plt.axis('equal')
plt.subplot(1, 2, 2)
plt.plot(Zprime[0, :], Zprime[1, :], 'ro', mfc = 'None', mec = 'r')
plt.plot(Zprimemapped[0, :], Zprimemapped[1, :], 'b.')
plt.legend(['actual points', 'reprojected points'])
plt.axis('equal')
plt.show()


# Print the geometric error
print(f'Geometric error: {geometric_error(H, Z, Zprime)}')

# Print the algebraic error
# YOUR CODE HERE
raise NotImplementedError()

### 6.1. Minimising the geometric error

In this course we chiefly discuss how to minimise the algebraic error.
The geometric error can of course also be minimised, although it requires non-linear iterative optimisation methods.
In the next cell, we use the default method from <code>scipy.optimize</code> to compute a homography that minimises the geometric error.
Compare the values of the geometric and algebraic cost functions to those in the previous cell.

In [ ]:
### Load optimisation module from scipy.
import scipy.optimize

# Find a homography that minimises the geometric cost function
opt = scipy.optimize.minimize(lambda h: geometric_error(h.reshape((3, 3), order = 'F'), Z, Zprime), H.flatten(order = 'F'))
Hgeom = opt['x'].reshape((3, 3), order = 'F')

# Print the geometric error
print(f'Geometric error: {geometric_error(Hgeom, Z, Zprime)}')

# Print the algebraic error
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# This cell was used for verification of labs in pandemic mode. Please ignore.